In [1]:
from core import (
    skf,
    X_train,
    Y_train,
    evaluate_model,
    save_model,
    save_results,
    score
)

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV


mlp_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        random_state=42,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10
    ))
])

param_dist_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (100, 50), (200, 100)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001, 0.01],
    'mlp__learning_rate': ['constant', 'adaptive'],
    'mlp__learning_rate_init': [0.001, 0.005, 0.01],
}


random_mlp = RandomizedSearchCV(
    estimator=mlp_pipe,
    param_distributions=param_dist_mlp,
    n_iter=30,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_mlp.fit(X_train, Y_train)

best_mlp = random_mlp.best_estimator_

print(f'Лучшие параметры MLP: {random_mlp.best_params_}')
print(f'Лучшая ROC_AUC: {random_mlp.best_score_:.4f}')

results_mlp = evaluate_model(
    model=best_mlp,
    X=X_train,
    y=Y_train,
    cv=skf,
    scoring_dict=score
)

save_results(
    results_dict=results_mlp,
    model_name='MLP'
)

save_model(
    model=best_mlp,
    model_name='MLP'
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Лучшие параметры MLP: {'mlp__learning_rate_init': 0.005, 'mlp__learning_rate': 'constant', 'mlp__hidden_layer_sizes': (200, 100), 'mlp__alpha': 0.0001, 'mlp__activation': 'relu'}
Лучшая ROC_AUC: 0.8678
Результаты MLP сохранены
Модель MLP сохранена


In [2]:
import pandas as pd


print(pd.read_csv('results_all.csv'))

      Model  Accuracy      F1  ROC-AUC  Precision           Saved_Time
0  CatBoost    0.8294  0.7817   0.8830     0.7671  2026-06-04 13:00:18
1   XGBoost    0.8395  0.7935   0.8870     0.7834  2026-06-04 14:53:42
2       MLP    0.8159  0.7310   0.8678     0.8344  2026-06-10 15:22:05
